# AI Product Analyst Agent

## Project Overview

This project develops an AI-powered Product Analyst Agent that uses Large Language Models (LLMs) to analyze customer feedback and generate actionable product insights.

The agent simulates a product analyst workflow by:

- Understanding customer complaints
- Identifying issue categories
- Evaluating priority levels
- Summarizing user problems
- Providing product improvement recommendations

The goal is to demonstrate how AI agents can support product management and customer experience optimization.

## Business Scenario

Product teams receive large amounts of unstructured customer feedback every day.

Manually analyzing these messages is time-consuming and difficult to scale.

This AI Product Analyst Agent aims to improve the workflow by automatically transforming raw customer feedback into structured product insights.

## 2. Environment Setup

This section imports required libraries and prepares the development environment.

The project uses:

- Python for implementation
- Pandas for data processing
- OpenAI-compatible API interface for LLM interaction
- dotenv for secure API key management

In [13]:
import os
import pandas as pd
from dotenv import load_dotenv

print("Environment ready")

Environment ready


## 3. Initialize LLM Client

The agent connects to a Large Language Model through an API interface.

API credentials are stored in environment variables instead of being directly written into the notebook to protect sensitive information.

In [14]:
from openai import OpenAI

load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

print("LLM client initialized")

LLM client initialized


### API Connection Test

A simple request is sent to verify whether the LLM connection works correctly.

In [15]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {
            "role": "user",
            "content": "Hello, introduce yourself briefly."
        }
    ]
)

print(response.choices[0].message.content)

Hello! I'm DeepSeek, an AI assistant created by DeepSeek (深度求索) — a Chinese company focused on artificial intelligence research. 

Here's a quick rundown of what I can do:

- **Knowledge & Conversations**: I can answer questions, explain concepts, help with writing, coding, brainstorming, and more.
- **Large Context**: I handle up to 1M tokens in a single conversation — that's enough to process entire books or long documents.
- **File Support**: You can upload images, PDFs, Word docs, Excel files, and more; I'll read and extract text from them.
- **Web Search**: I support web search, but you'll need to manually enable it in the web or app interface.
- **Multilingual**: I respond in whatever language you use.
- **Current Models**: I'm the latest DeepSeek model, with a knowledge cutoff in May 2025.

I'm completely free to use, available via web, iOS, and Android apps, and I don't require any fees or subscriptions. 

What would you like to talk about or explore today? 😊


## 4. Define AI Product Analyst Agent

This section implements the core logic of the AI Product Analyst Agent.

The agent acts as a product analyst by transforming unstructured customer feedback into structured insights.

The analysis framework includes:

- Issue identification
- Problem categorization
- User impact assessment
- Priority evaluation
- Product improvement suggestions

The LLM is instructed to respond from a product management perspective rather than providing a general summary.

In [16]:
def analyze_feedback(feedback):

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": """
You are an AI Product Analyst Agent.

Your task is to analyze customer feedback and generate actionable product insights.

For each feedback item, provide:

1. Main issue
2. Issue category
3. User impact
4. Priority level (High/Medium/Low)
5. Recommended product action

Answer from a product manager perspective.
"""
            },
            {
                "role": "user",
                "content": feedback
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

## 5. Test the Agent

A single customer feedback example is used to verify whether the agent can correctly interpret user problems and generate product insights.

This step demonstrates the basic reasoning capability of the AI Product Analyst Agent before applying it to larger feedback datasets.

In [17]:
test_feedback = """
The app crashes every time I upload photos.
"""

result = analyze_feedback(test_feedback)

print(result)

Here is the analysis for the feedback item: "The app crashes every time I upload photos."

---

### Feedback Analysis Report

**Feedback:** "The app crashes every time I upload photos."

---

**1. Main Issue**
The application experiences a fatal, reproducible crash (force close) triggered specifically by the photo upload function. This is a critical stability defect, not a usability preference.

**2. Issue Category**
- **Primary:** Technical Bug / Stability (Crash)
- **Secondary:** Core Feature Failure (Upload functionality)

**3. User Impact**
- **Severity:** Critical. The user is completely blocked from performing a core action (uploading photos).
- **Frustration:** Extremely high. The user cannot complete their primary task, leading to data loss (if the photo isn't saved) and immediate abandonment of the app.
- **Trust:** Damages user trust in the app's reliability, leading to negative reviews and churn.
- **Scope:** The word "every time" suggests a 100% reproduction rate for this u

## 6. Batch Customer Feedback Analysis

Real product teams usually receive multiple customer feedback messages rather than a single complaint.

This section extends the agent from single feedback analysis to batch processing.

The agent analyzes multiple feedback items and converts unstructured user comments into structured product insights.

In [18]:
feedback_list = [
    "The app crashes every time I upload photos.",
    "Battery drains too fast after the latest update.",
    "Customer service response is extremely slow.",
    "I love the new design, but navigation is confusing.",
    "The subscription price is too expensive."
]


feedback_df = pd.DataFrame({
    "feedback": feedback_list
})


feedback_df

,feedback
0,The app crashes every time I upload photos.
1,Battery drains too fast after the latest update.
2,Customer service response is extremely slow.
3,"I love the new design, but navigation is confu..."
4,The subscription price is too expensive.


## Apply Agent to Multiple Feedback Items

Each feedback item is processed independently by the AI Product Analyst Agent.

The output will be stored together with the original customer feedback for further analysis.

In [19]:
analysis_results = []


for feedback in feedback_df["feedback"]:
    result = analyze_feedback(feedback)
    analysis_results.append(result)


feedback_df["analysis"] = analysis_results


feedback_df

,feedback,analysis
0,The app crashes every time I upload photos.,**1. Main Issue** \nThe application has a cri...
1,Battery drains too fast after the latest update.,**1. Main Issue:** \nThe latest software upda...
2,Customer service response is extremely slow.,**1. Main Issue** \nCustomers are experiencin...
3,"I love the new design, but navigation is confu...",**Analysis of Customer Feedback:**\n\n**Feedba...
4,The subscription price is too expensive.,"**Feedback Analysis: ""The subscription price i..."


## 7. Structured Product Insight Generation

For practical product analytics workflows, structured outputs are easier to store, compare, and visualize.

This section modifies the agent to generate structured JSON-style insights containing:

- Issue
- Category
- Severity
- Priority
- Recommended action

In [20]:
def analyze_feedback_structured(feedback):

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": """
You are an AI Product Analyst.

Analyze customer feedback and return ONLY JSON format.

Required fields:

{
"issue": "",
"category": "",
"severity": "",
"priority": "",
"recommendation": ""
}

Do not include markdown.
"""
            },
            {
                "role": "user",
                "content": feedback
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

## Test Structured Output

A structured response allows downstream systems to store and analyze AI-generated product insights.

In [21]:
structured_result = analyze_feedback_structured(
    "The subscription price is too expensive."
)

print(structured_result)

{
"issue": "The subscription price is too expensive.",
"category": "Pricing",
"severity": "High",
"priority": "High",
"recommendation": "Review pricing strategy, consider offering tiered plans, discounts, or a more affordable option to improve customer satisfaction and retention."
}


## 8. Convert AI Insights into Structured Data

The generated AI insights are converted into a dataframe format.

This allows product teams to perform further analysis, reporting, and visualization.

In [22]:
import json


structured_outputs = []


for feedback in feedback_list:

    result = analyze_feedback_structured(feedback)

    structured_outputs.append(
        json.loads(result)
    )


insight_df = pd.DataFrame(structured_outputs)


insight_df

,issue,category,severity,priority,recommendation
0,The app crashes every time the user uploads ph...,Stability,High,Critical,Investigate and fix the crash in the photo upl...
1,Battery drains too fast after the latest update.,Performance,High,High,Investigate the latest update for battery cons...
2,Customer service response is extremely slow.,Customer Support,High,High,Investigate support ticket routing and staffin...
3,Navigation is confusing despite liking the new...,Usability,Medium,High,Conduct usability testing to identify specific...
4,The subscription price is too expensive.,Pricing,Medium,High,"Review pricing strategy, consider offering tie..."


## 9. Export AI Generated Insights

The final insights are exported as CSV files.

This demonstrates how the AI Product Analyst Agent can be integrated into a real product analytics workflow.

In [23]:
import os


output_dir = "outputs"

os.makedirs(output_dir, exist_ok=True)


insight_df.to_csv(
    os.path.join(output_dir, "product_insights.csv"),
    index=False
)


print("Export completed successfully.")

Export completed successfully.


# 10. Conclusion

This project demonstrates how AI Agents can support product management workflows by transforming unstructured customer feedback into actionable insights.

The AI Product Analyst Agent is able to:

- Interpret customer problems
- Categorize product issues
- Evaluate user impact
- Prioritize product improvements
- Generate structured recommendations

Future improvements could include:

- Connecting with real customer feedback databases
- Adding retrieval augmented generation (RAG)
- Integrating product metrics and user behavior data
- Building multi-agent collaboration workflows

# 11. AI Priority Ranking Agent

Product teams need to decide which problems should be solved first.

This section introduces a priority ranking agent that evaluates issues based on:

- User impact
- Severity
- Business importance
- Recommended priority order

The agent simulates a product manager's backlog prioritization process.

In [24]:
def prioritize_issues(insights):

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": """
You are a senior product manager.

Your task is to prioritize product issues.

Rank issues according to:

1. User impact
2. Severity
3. Business importance
4. Implementation urgency

Return JSON format:

[
{
"rank":1,
"issue":"",
"reason":"",
"priority":"High/Medium/Low"
}
]

Only return JSON.
"""
            },
            {
                "role":"user",
                "content":str(insights)
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

## Test Priority Ranking Agent

The priority agent receives multiple AI-generated insights and creates a ranked product backlog.


In [25]:
priority_result = prioritize_issues(
    insight_df.to_dict("records")
)


print(priority_result)

[
{
"rank": 1,
"issue": "The app crashes every time the user uploads photos.",
"reason": "Critical stability issue directly blocks a core feature, causing immediate user frustration and potential data loss; highest user impact and severity, urgent hotfix needed.",
"priority": "High"
},
{
"rank": 2,
"issue": "Battery drains too fast after the latest update.",
"reason": "High severity performance regression affects all users post-update, leading to negative reviews and churn; requires urgent investigation and patch to restore trust.",
"priority": "High"
},
{
"rank": 3,
"issue": "Customer service response is extremely slow.",
"reason": "High severity support issue damages customer satisfaction and retention, impacting business reputation; urgent to implement SLAs and self-service options to reduce load.",
"priority": "High"
},
{
"rank": 4,
"issue": "Navigation is confusing despite liking the new design.",
"reason": "Medium severity usability issue affects user engagement and task completi